In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from IPython.display import clear_output
import os
!git clone --filter=blob:none --no-checkout https://github.com/spMohanty/PlantVillage-Dataset.git
%cd PlantVillage-Dataset
!git sparse-checkout init --cone
!git sparse-checkout set raw
!git checkout
from google.colab import output
!mkdir -p /content/drive/MyDrive/Plant-village-ds
!zip -r color.zip /content/PlantVillage-Dataset/raw/color
!cp color.zip /content/drive/MyDrive/Plant-village-ds/
output.clear()

In [ ]:
!unzip /content/drive/MyDrive/Plant-village-ds/color.zip -d /content/
clear_output()

Archive:  /content/drive/MyDrive/Plant-village-ds/color.zip
replace /content/content/PlantVillage-Dataset/raw/color/Apple___Cedar_apple_rust/dc75b693-0cd6-4a48-b0a0-e442070df2cc___FREC_C.Rust 9951.JPG? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

NameError: name 'clear_output' is not defined

In [ ]:
import tensorflow as tf
import os
import shutil

In [ ]:
data_set = '/content/content/PlantVillage-Dataset/raw/color/'
filtered_path = '/content/Filtered_PlantVillage/'

# Create new directory
os.makedirs(filtered_path, exist_ok=True)

for class_name in os.listdir(data_set):
    # Skip healthy classes
    if "healthy" in class_name.lower():
        continue

    src = os.path.join(data_set, class_name)
    dst = os.path.join(filtered_path, class_name)

    if os.path.isdir(src):
        shutil.copytree(src, dst)


FileExistsError: [Errno 17] File exists: '/content/Filtered_PlantVillage/Apple___Cedar_apple_rust'

In [ ]:
img_size = (224, 224)
batch_size = 32

training_set= tf.keras.utils.image_dataset_from_directory(
    filtered_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

validation_set = tf.keras.utils.image_dataset_from_directory(
    filtered_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

Found 39221 files belonging to 26 classes.
Using 31377 files for training.
Found 39221 files belonging to 26 classes.
Using 7844 files for validation.


In [ ]:
import tensorflow as tf
AUTOTUNE = tf.data.AUTOTUNE

# Remove .cache() for now, as it might be causing memory issues with a large dataset
training_set = training_set.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_set = validation_set.prefetch(buffer_size=AUTOTUNE)

In [ ]:
from tensorflow import keras
from tensorflow.keras.layers import Rescaling
training_set = training_set.map(lambda x, y: (Rescaling(1./255)(x), y))
validation_set = validation_set.map(lambda x, y: (Rescaling(1./255)(x), y))

In [ ]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

training_set = training_set.map(lambda x, y: (data_augmentation(x, training=True), y))

In [ ]:
from tensorflow.keras.layers import Dense,Conv2D,MaxPool2D,Flatten,Dropout,BatchNormalization
from tensorflow.keras.models import Sequential
from keras.optimizers import AdamW, SGD

In [ ]:
model = Sequential([

    tf.keras.Input(shape=(224, 224, 3)),
    Conv2D(32, 3, activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(32, 3, activation='relu'),
    BatchNormalization(),
    MaxPool2D(2,2),

    Conv2D(64, 3, activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, 3, activation='relu'),
    BatchNormalization(),
    MaxPool2D(2,2),

    Conv2D(128, 3, activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, 3, activation='relu'),
    BatchNormalization(),
    MaxPool2D(2,2),

    Conv2D(256, 3, activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, 3, activation='relu'),
    BatchNormalization(),
    MaxPool2D(2,2),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(12, activation='softmax') # Changed from 26 to 12
])

In [ ]:
model.compile(
    optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-5),
    loss='sparse_categorical_crossentropy', # Changed to sparse_categorical_crossentropy
    metrics=['accuracy']
)

In [ ]:
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 222, 222, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 111, 111, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 111, 111, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 54, 54, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 54, 54, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 26, 26, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             

 Total params: 20,057,132 (76.51 MB)

 Trainable params: 20,055,212 (76.50 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [ ]:
import sys
from io import StringIO

stringlist = []
model.summary(print_fn=lambda x: stringlist.append(x))
short_model_summary = "\n".join(stringlist)

with open("/content/drive/MyDrive/model_summary.txt", "w") as f:
    f.write(short_model_summary)

print("Model summary saved to /content/drive/MyDrive/models/model_summary.txt")

Model summary saved to /content/drive/MyDrive/models/model_summary.txt


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6),
    ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)
]

In [ ]:
history = model.fit(
    training_set,
    validation_data=validation_set,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10


In [ ]:
train_loss,train_acc = model.evaluate(training_set)

1501/1501 ━━━━━━━━━━━━━━━━━━━━ 838s 558ms/step - accuracy: 0.9538 - loss: 0.1478


In [ ]:
val_loss,val_acc = model.evaluate(validation_set)

375/375 ━━━━━━━━━━━━━━━━━━━━ 18s 48ms/step - accuracy: 0.9112 - loss: 0.3183


In [ ]:
model.save("/content/drive/MyDrive/models/model26.keras")

In [ ]:
model = tf.keras.models.load_model("/content/drive/MyDrive/models/model26.keras")

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = '/content/drive/MyDrive/path/to/your/image.jpg'

img = image.load_img(img_path, target_size=(224, 224))

img_array = image.img_to_array(img)


img_array = np.expand_dims(img_array, axis=0)

preprocessed_image = img_array / 255.0

In [ ]:
prediction = model.predict(preprocessed_image)
prediction,prediction.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


(array([[1.9965641e-14, 2.6980317e-07, 5.8268140e-11, 8.5792884e-08,
         2.0488612e-06, 5.9247550e-06, 9.9291261e-05, 7.3190132e-04,
         6.4906466e-04, 2.6217925e-10, 2.2102043e-04, 4.0047343e-07,
         3.0101359e-07, 6.8552490e-11, 7.9553730e-10, 2.1691960e-06,
         1.3477934e-13, 5.4010257e-10, 1.3982147e-10, 6.8971198e-09,
         6.4505815e-07, 9.9828678e-01, 5.2408352e-11, 7.6869334e-08,
         2.8035230e-10, 1.8401529e-10]], dtype=float32),
 (1, 26))

In [ ]:
result_index = np.argmax(prediction)
result_index

np.int64(21)

In [ ]:
class_names[result_index]

'Potato___Early_blight'